# Flower vs DPS - Figure qualitative
Ce notebook genere uniquement la figure qualitative (Jules et Hamza) a partir des sorties de run_compare.sh.
Les scores PSNR/SSIM sont affiches dans les titres des colonnes Flower et DPS.

In [10]:
from pathlib import Path
import json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image

# Robust path resolution whether cwd is the notebook folder or workspace root
cwd = Path.cwd()
candidate_dirs = [
    cwd / "results",
    cwd / "MVA" / "compare_with_DPS" / "results",
]
results_dir = None
for c in candidate_dirs:
    if (c / "summary.json").exists():
        results_dir = c
        break
if results_dir is None:
    raise FileNotFoundError(
        "summary.json not found. Expected either './results/summary.json' or './MVA/compare_with_DPS/results/summary.json'."
    )

summary_path = results_dir / "summary.json"
fig_dir = results_dir / "report_figures"
fig_dir.mkdir(parents=True, exist_ok=True)

with open(summary_path, "r", encoding="utf-8") as f:
    summary = json.load(f)

flower = summary["flower"]
dps = summary["dps"]

paired_dir = results_dir / "paired"
ids = sorted([p.name for p in paired_dir.iterdir() if p.is_dir()])
n = len(ids)
assert n > 0, "No paired results found. Run run_compare.sh first."

if n == 2:
    row_titles = ["Jules", "Hamza"]
else:
    row_titles = [f"Image {im_id}" for im_id in ids]

col_titles = [
    "Ground truth",
    f"Flower reconstruction\nPSNR {flower['psnr_rec']:.2f} | SSIM {flower['ssim_rec']:.3f}",
    f"DPS reconstruction\nPSNR {dps['psnr_rec']:.2f} | SSIM {dps['ssim_rec']:.3f}",
]
cols = ["label.png", "flower_recon.png", "dps_recon.png"]

fig, axes = plt.subplots(n, len(cols), figsize=(11.5, 2.9 * n), dpi=160)
if n == 1:
    axes = np.array([axes])

for r, im_id in enumerate(ids):
    for c, (fname, title) in enumerate(zip(cols, col_titles)):
        ax = axes[r, c]
        img = np.array(Image.open(paired_dir / im_id / fname).convert("RGB"))
        ax.imshow(img)
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.6)
            spine.set_color("#D0D0D0")
        if r == 0:
            ax.set_title(title, fontsize=10, pad=8)
        if c == 0:
            ax.set_ylabel(row_titles[r], fontsize=11)

fig.suptitle("Flower vs DPS on OOD images (Jules, Hamza)", fontsize=13, y=0.995)
fig.tight_layout()
fig.subplots_adjust(top=0.84)

fig_path = fig_dir / "figure_qualitative_flower_vs_dps_jules_hamza.png"
fig.savefig(fig_path, bbox_inches="tight", facecolor="white")
plt.show()
plt.close(fig)

fig_path

PosixPath('/users/eleves-a/2022/jules.damidaux/Documents/Flower/MVA/compare_with_DPS/results/report_figures/figure_qualitative_flower_vs_dps_jules_hamza.png')